# 使用 AgentCore Payments 构建能够为 API 付费的 AI 智能体

AI 智能体常常需要从外部服务获取信息才能完成任务。采购智能体可能需要在制裁数据库中核查供应商、获取最新的风险报告，或使用付费搜索 API。当这些服务按请求收费时，应用需要明确的规则，规定智能体可以请求哪些购买，以及由谁来批准支出。

智能体商务（Agentic commerce）指的是这样一类工作流：智能体可以请求付费资源，而支付的决定权仍由应用代码掌握。Amazon Bedrock AgentCore Payments 提供了有边界的支付会话基础设施，OpenAI Agents SDK 则允许模型请求调用由应用定义的工具。在发生任何支付之前，应用会先检查商家、用途、金额和审批。

本 notebook 通过一个供应商调研示例来讲解这一工作流。内容涵盖 AgentCore Payments 流程、x402 支付交换、由应用掌控的支出控制，以及收据验证。同样的支付控制会先通过本地模拟进行演示，然后再连接到 AWS 测试网上的 AgentCore Payments。


## 为什么智能体需要访问付费服务

设想一个采购智能体正在审查一家名为 Northstar Components 的虚构供应商。智能体掌握了供应商的基本信息，但在为人工审核者总结潜在问题之前，它需要从付费 API 获取一份最新的风险报告。

应用批准了该数据提供方，定义了供应商调研这一用途，并为报告设定了最高价格。它还会记录与该请求关联的人工审批。当模型判断这份报告有用时，它会携带供应商报告 URL 和已批准的用途请求调用 `x402_fetch` 工具。

商家返回一个 HTTP `402 Payment Required` 质询，说明其支付条款。应用将这些条款与已批准的商家、金额、网络、有效期和预算进行比对。如果请求通过检查，应用会获取一个有边界的支付证明，取回报告，并记录收据和审计历史。随后，智能体根据工具返回的报告和收据证据准备回答。


## 架构与支付权限

该系统划分了四项职责：

1. **AI 智能体**识别付费服务何时有助于完成用户请求，并调用由应用定义的工具。
2. **应用**在记录收据和审计历史之前，检查商家、购买用途、金额、审批和支出限额。
3. **AgentCore Payments** 在已批准支付会话的限额内生成支付证明。
4. **付费 API** 检查支付证明并返回所请求的内容。

本地示例会模拟 AgentCore Payments 这一步。连接 AWS 的工作流则使用 AgentCore Payments 生成有边界的证明。

应用是模型与支付基础设施之间的边界。模型接收资源 URL 和调研用途，而审批记录、支出限额、支付凭据和审计状态都不在它的控制范围之内。

![智能体通过应用控制和 AgentCore Payments 请求付费的供应商报告](../../../../images/partners/AWS/controlled-agentic-commerce-overview.png)

*应用掌控审批和支出，而 AgentCore Payments 为连接 AWS 的工作流提供有边界的支付会话。*


## 1. 设置供应商调研示例

本示例需要 Python 3.11 至 3.13 以及 `uv`。请按照 `README.md` 中在仓库根目录运行 Jupyter 的命令操作，以便 notebook 能够加载其软件包并显示架构图。

第一部分使用脚本化模型、模拟商家和合成的支付证明。这种设置让你无需配置 AWS 凭据或转移资金，就能检查同样的请求、审批、收据和审计流程。最后一节会展示真实的 Amazon Bedrock 模型和 AgentCore Payments 测试网资源在工作流中的接入位置。

下面的单元格会定位示例软件包，使其应用、商家和策略模块可在 notebook 中使用。


In [ ]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd / "examples/partners/AWS/controlled_agentic_commerce_with_agentcore_payments",
    *cwd.parents,
]
ROOT = next(
    (path for path in candidates if (path / "src/agentic_commerce").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from its example directory.")
sys.path.insert(0, str(ROOT / "src"))

print(f"Example root: {ROOT}")

## 2. 定义供应商请求和支出限额

在模型可以请求工具之前，应用需要先确立支付规则。在本示例中，供应商报告的价格为 0.25 个合成 USDC。应用还定义了已批准的商家、允许的调研用途、本次购买的最高金额，以及本次智能体运行可用的总预算。

`ApprovalGrant` 将购买与特定的请求 ID 和资源 URL 绑定。其有效期限制了该授权保持有效的时长。模型从不提供或修改这些值；它们由应用代码在创建工具时提供。


In [ ]:
from datetime import UTC, datetime, timedelta
from decimal import Decimal

from agentic_commerce.demo import build_demo
from agentic_commerce.merchant import RESOURCE_URL
from agentic_commerce.models import ApprovalGrant

now = datetime.now(UTC)
application, merchant, local_payments = build_demo(now)
approval = ApprovalGrant(
    approval_id="approval-cookbook-001",
    request_id="request-cookbook-001",
    resource_url=RESOURCE_URL,
    purpose="supplier_due_diligence",
    maximum_amount=Decimal("0.25"),
    approved_by="synthetic-cookbook-reviewer",
    approved_at=now,
    expires_at=now + timedelta(minutes=10),
)

{
    "approved_merchant": str(RESOURCE_URL),
    "purpose": approval.purpose,
    "approved_amount": str(approval.maximum_amount),
    "run_budget": str(application.policy.policy.per_run_limit),
    "currency": approval.currency,
    "approval_expires_at": approval.expires_at.isoformat(),
}

## 3. 为已批准的购买定义智能体工具

`x402_fetch` 函数是智能体唯一可用的与支付相关的工具。模型可以提供资源 URL 和业务用途，但请求 ID、幂等键、支出策略和审批授权由应用提供。

在开始购买之前，该工具会消耗一个一次性能力（capability）。这项检查可防止模型在同一次智能体运行中发起多个涉及资金的请求。随后，`application.purchase` 会先验证商家和支付质询，再生成收据。

`AgentPurchaseEvidence` 只返回模型准备回答所需的报告和收据字段。钱包材料、支付证明请求头以及完整的应用审计记录都不会进入模型上下文。


In [ ]:
from threading import Lock

from agents import Agent, ModelSettings, function_tool
from openai.types.shared import Reasoning

from agentic_commerce.agent import PurchaseResultRecorder, SupplierResearchOutput
from agentic_commerce.errors import AgentResultInvalid
from agentic_commerce.learning_model import ScriptedCommerceLearningModel
from agentic_commerce.models import AgentPurchaseEvidence, PurchaseRequest

recorder = PurchaseResultRecorder()
model = ScriptedCommerceLearningModel()
tool_capability = {"used": False}
tool_lock = Lock()


@function_tool
def x402_fetch(resource_url: str, purpose: str) -> str:
    """Fetch one paid resource using application-owned approval."""

    with tool_lock:
        if tool_capability["used"]:
            raise AgentResultInvalid(
                "access_count_invalid",
                "The economic tool permits exactly one invocation.",
            )
        tool_capability["used"] = True

    purchase = application.purchase(
        PurchaseRequest(
            request_id="request-cookbook-001",
            resource_url=resource_url,
            purpose=purpose,
            idempotency_key="purchase-cookbook-001",
        ),
        approval=approval,
    )
    recorder.record(purchase)
    return AgentPurchaseEvidence(
        status=purchase.status,
        report=purchase.report,
        receipt_id=purchase.receipt.receipt_id,
        amount=purchase.receipt.amount,
        currency=purchase.receipt.currency,
        requires_human_approval=purchase.authorization.requires_human_approval,
    ).model_dump_json()


agent = Agent(
    name="Supplier research agent",
    model=model,
    model_settings=ModelSettings(reasoning=Reasoning(effort="low"), store=False),
    output_type=SupplierResearchOutput,
    tools=[x402_fetch],
    instructions=(
        "Call x402_fetch exactly once for the approved supplier report. "
        "Return only the typed summary supported by its receipt."
    ),
)

{"agent": agent.name, "tools": [tool.name for tool in agent.tools]}

## 4. 运行 x402 支付工作流

x402 使用 HTTP `402 Payment Required` 响应来传达付费请求的条款。当应用首次请求供应商报告时，商家会返回一个质询，其中包含金额、资产、网络、收款方和有效期。

应用会将这些条款与其已批准的策略进行比对。如果商家、价格、请求用途和人工审批都有效，AgentCore Payments 会在连接 AWS 的工作流中创建一个有边界的证明。本地示例则使用合成证明来演示同样的流程。应用携带该证明重试请求，商家随即以 HTTP `200` 返回供应商报告。

![智能体、应用、商家与 AgentCore Payments 之间的 x402 请求时序](../../../../images/partners/AWS/controlled-agentic-commerce-x402-sequence.png)

*商家的 402 响应提供支付条款。应用审批发生在请求支付证明之前。*

`Runner.run` 执行 Agents SDK 循环。脚本化模型在第一轮请求调用 `x402_fetch`，并在工具返回后准备一份带类型的供应商摘要。应用会将该摘要与其实际记录的购买进行核对。


In [ ]:
from agents import RunConfig, Runner

from agentic_commerce.agent import (
    DEFAULT_PROMPT,
    SupplierResearchRun,
    validate_supplier_research_output,
)

result = await Runner.run(
    agent,
    DEFAULT_PROMPT,
    max_turns=4,
    run_config=RunConfig(
        tracing_disabled=True,
        workflow_name="Synthetic Agentic Commerce",
    ),
)
purchase = validate_supplier_research_output(result.final_output, recorder)
run = SupplierResearchRun(output=result.final_output, purchase=purchase)

{
    "model_turns": model.model_turns,
    "tool_calls": model.requested_tools,
    "merchant_requests": merchant.request_count,
    "synthetic_charges": local_payments.charge_count,
    "model_inference_calls": 0,
    "aws_calls": 0,
    "value_transferred": False,
}

### 直接使用 Responses API

在前面的示例中，由 OpenAI Agents SDK 协调工具循环，但支付工作流并不依赖这一编排层。Responses API 可以将 `x402_fetch` 作为函数暴露出来，返回模型提议的参数，并在应用处理完购买后接收 `function_call_output`。

应用仍会在付款之前检查商家、用途、审批和支出限额。在连接 AWS 的工作流中，AgentCore Payments 仍负责生成有边界的证明。下面的示例使用与本地演练相同的模拟商家和支付处理器。

这种替代方案会调用一个真实的、托管在 Bedrock 上的 GPT-5.6 Sol 模型，因此只有在设置了 `ALLOW_PAID_INFERENCE=1` 并配置了有效的 AWS 凭据时才会运行。它不会启用实时支付，也不会转移资金。


In [ ]:
import json
import os

from openai import OpenAI
from openai.providers import bedrock

responses_tool = {
    "type": "function",
    "name": "x402_fetch",
    "description": "Fetch one approved supplier report.",
    "parameters": {
        "type": "object",
        "properties": {
            "resource_url": {"type": "string"},
            "purpose": {"type": "string"},
        },
        "required": ["resource_url", "purpose"],
        "additionalProperties": False,
    },
    "strict": True,
}
responses_result = {
    "result": "SKIPPED",
    "reason": "Set ALLOW_PAID_INFERENCE=1 to call the Bedrock model.",
    "model_calls": 0,
    "live_payment_calls": 0,
}

if os.environ.get("ALLOW_PAID_INFERENCE") == "1":
    responses_client = OpenAI(
        provider=bedrock(
            region=os.environ.get("AWS_REGION", "us-east-2"),
            profile=os.environ.get("BEDROCK_AWS_PROFILE") or None,
            api_key=None,
        )
    )
    prompt = (
        f"Call x402_fetch for {RESOURCE_URL} with purpose "
        "supplier_due_diligence. After the tool returns, provide JSON "
        "containing supplier, receipt_id, amount, and currency."
    )
    first_response = responses_client.responses.create(
        model="openai.gpt-5.6-sol",
        input=prompt,
        tools=[responses_tool],
        tool_choice={"type": "function", "name": "x402_fetch"},
        parallel_tool_calls=False,
        reasoning={"effort": "low"},
        max_output_tokens=128,
        store=False,
    )
    requested_tools = [
        item for item in first_response.output if item.type == "function_call"
    ]
    if len(requested_tools) != 1 or requested_tools[0].name != "x402_fetch":
        raise AgentResultInvalid(
            "access_count_invalid", "Expected exactly one x402_fetch request."
        )

    arguments = json.loads(requested_tools[0].arguments)
    responses_now = datetime.now(UTC)
    responses_application, responses_merchant, responses_payments = build_demo(
        responses_now
    )
    responses_request_id = "request-responses-cookbook-001"
    responses_approval = approval.model_copy(
        update={
            "approval_id": "approval-responses-cookbook-001",
            "request_id": responses_request_id,
            "approved_at": responses_now,
            "expires_at": responses_now + timedelta(minutes=10),
        }
    )
    responses_purchase = responses_application.purchase(
        PurchaseRequest(
            request_id=responses_request_id,
            resource_url=arguments["resource_url"],
            purpose=arguments["purpose"],
            idempotency_key="purchase-responses-cookbook-001",
        ),
        approval=responses_approval,
    )
    responses_evidence = AgentPurchaseEvidence(
        status=responses_purchase.status,
        report=responses_purchase.report,
        receipt_id=responses_purchase.receipt.receipt_id,
        amount=responses_purchase.receipt.amount,
        currency=responses_purchase.receipt.currency,
        requires_human_approval=(
            responses_purchase.authorization.requires_human_approval
        ),
    )
    response_input = [
        {"role": "user", "content": prompt},
        *(
            item.model_dump(mode="json", exclude_none=True)
            for item in first_response.output
        ),
        {
            "type": "function_call_output",
            "call_id": requested_tools[0].call_id,
            "output": responses_evidence.model_dump_json(),
        },
    ]
    final_response = responses_client.responses.create(
        model="openai.gpt-5.6-sol",
        input=response_input,
        tools=[responses_tool],
        tool_choice="none",
        parallel_tool_calls=False,
        reasoning={"effort": "low"},
        max_output_tokens=256,
        store=False,
    )
    final_output = json.loads(final_response.output_text)
    expected_output = {
        "supplier": responses_purchase.report.supplier,
        "receipt_id": responses_purchase.receipt.receipt_id,
        "amount": str(responses_purchase.receipt.amount),
        "currency": responses_purchase.receipt.currency,
    }
    if any(final_output.get(key) != value for key, value in expected_output.items()):
        raise AgentResultInvalid(
            "agent_output_mismatch", "The model response did not match its receipt."
        )
    responses_result = {
        "result": "PASSED",
        "responses_api_calls": 2,
        "tool_calls": [requested_tools[0].name],
        "merchant_requests": responses_merchant.request_count,
        "synthetic_charges": responses_payments.charge_count,
        "receipt_verified": True,
        "live_payment_calls": 0,
        "value_transferred": False,
    }

responses_result

## 5. 验证供应商报告、收据和审计历史

模型返回结构化的供应商摘要，但有效的 schema 只能说明响应具有预期的结构。应用仍需检查所报告的收据、金额、币种和供应商详情是否与已完成的工具请求相符。

第一个单元格将智能体提议的结果与应用掌握的收据进行比较。第二个单元格展示审计流程：从最初的资源请求，到授权、支付证明创建、向商家重试，再到返回内容。


In [ ]:
proposal = run.output.model_dump(mode="json")
receipt = run.purchase.receipt.model_dump(mode="json")

{
    "typed_agent_proposal": proposal,
    "application_receipt": {
        "receipt_id": receipt["receipt_id"],
        "amount": receipt["amount"],
        "currency": receipt["currency"],
        "network": receipt["network"],
        "reused": receipt["reused"],
    },
}

In [ ]:
[
    {
        "sequence": event.sequence,
        "event_type": event.event_type.value,
    }
    for event in run.purchase.audit_events
]

## 6. 检查被拒绝的购买和伪造的收据

一个有用的支付示例应当展示当控制机制拒绝请求时会发生什么。第一项检查在缺少必需的人工审批的情况下，再次提交一个供应商报告请求。应用可以查看商家的质询，但必须在创建新的支付证明或产生扣费之前拒绝这次购买。

请求被拒绝后，现有的合成扣费次数应保持不变。


In [ ]:
from agentic_commerce.errors import PolicyDenied

blocked_request = PurchaseRequest(
    request_id="request-cookbook-002",
    resource_url=RESOURCE_URL,
    purpose="supplier_due_diligence",
    idempotency_key="purchase-cookbook-002",
)

try:
    application.purchase(blocked_request, approval=None)
except PolicyDenied as exc:
    blocked_result = {
        "status": "DENIED",
        "reason": exc.code,
        "synthetic_charges": local_payments.charge_count,
    }
else:
    raise AssertionError("Unapproved purchase unexpectedly succeeded")

blocked_result

第二项检查修改了一份原本有效的供应商摘要中的收据标识符。输出验证器会将该响应与应用实际观察到的购买进行比较，并拒绝不匹配的结果。

这两项检查覆盖了不同的边界：付款前的授权，以及模型响应后的证据验证。


In [ ]:
from agentic_commerce.agent import (
    PurchaseResultRecorder,
    validate_supplier_research_output,
)

fabricated = run.output.model_copy(update={"receipt_id": "receipt-fabricated"})
try:
    validate_supplier_research_output(
        fabricated,
        PurchaseResultRecorder(results=[run.purchase]),
    )
except AgentResultInvalid as exc:
    negative_result = {"status": "REJECTED", "code": exc.code}
else:
    raise AssertionError("Fabricated output unexpectedly passed validation")

negative_result

## 7. 将 AgentCore Payments 连接到 AWS 测试网

连接后的供应商调研工作流使用一个托管在 Bedrock 上的 OpenAI 模型、AgentCore Payments，以及 Base Sepolia 上一个已批准的 x402 商家。应用保留了本地模拟中演示过的同一套商家策略、支出审批、请求身份和收据检查。AgentCore Payments 会创建一个短时会话，并设有自己的最高支出限额。

![本地供应商调研与 AWS 测试网请求共享同一套应用审批和支出控制](../../../../images/partners/AWS/controlled-agentic-commerce-local-and-testnet.png)

*本地模拟与 AWS 测试网工作流共享同一套应用控制。在连接的环境中，AgentCore Payments 提供有边界的支付会话。*

在启用 AWS 路径之前，运维人员需要在 notebook 之外预先配置支付管理器、连接器、支付工具（instrument）以及针对确切钱包的签名权限。模型推理、会话管理和支付执行分别由不同的职责承担。`README.md` 介绍了设置方法、支持的配置、低于预算时的拒绝检查以及清理流程。

本 notebook 不使用 AgentCore Runtime。notebook 进程本身就是应用的运行时，除非运维人员明确启用，否则所有实时支付闸门都处于禁用状态。


### 在发起网络调用前检查测试网就绪状态

就绪检查只检查本地配置，不会联系 AWS、模型、商家或钱包。它会确认必需的支付启用开关都已设置、AWS 角色已分离、会话预算是有限的，并且本地状态文件中没有记录任何支付会话。

该检查还会拒绝与商家连接的 DNS 和 TLS 保护不兼容的代理设置。在默认配置下，预期结果是 `NOT_READY`，因为实时支付闸门处于禁用状态。


In [ ]:
from agentic_commerce.agentcore_e2e import (
    readiness_report,
    run_managed_e2e,
)

agentcore_readiness = readiness_report()
agentcore_readiness

### 检查支付工具和网络

当明确启用 `ALLOW_AGENTCORE_READ_ONLY=1` 时，应用可以在不创建支付会话的情况下检查已配置的支付工具。该支付工具必须处于 `ACTIVE` 状态，使用 `ETHEREUM` 钱包网络，并报告确切的 `BASE_SEPOLIA` 和 `USDC` 余额范围。

缺少闸门时返回 `SKIPPED`。网络或资产不匹配时返回 `NOT_READY`。这些检查不会暴露余额数额、生成支付证明、联系商家或转移价值。


In [ ]:
from agentic_commerce.agentcore_infrastructure import (
    inspect_payment_instrument,
)

agentcore_infrastructure = inspect_payment_instrument()
agentcore_infrastructure

### 运行一笔已批准的测试网交易

只有在检查了已批准的商家、收款方、资产、金额、钱包以及短时会话配置之后，运维人员才能启用托管的测试网运行。该工作流要求全部五个明确的闸门：`RUN_AGENTCORE_E2E`、`ALLOW_AGENTCORE_SESSION_ADMIN`、`ALLOW_PAID_INFERENCE`、`ALLOW_AGENTCORE_TESTNET` 和 `APPROVE_AGENTCORE_TESTNET_PURCHASE`。

组合执行路径会在开始模型推理或支付活动之前再次检查这些闸门。它会创建一个有边界的会话，将智能体的请求交由应用策略处理，请求一个 AgentCore 支付证明，并向已批准的商家重试一次。

会话创建是串行执行的。如果本地会话持久化失败，应用会尝试执行补偿性清理。无论托管运行成功还是失败，都会在 `finally` 中尝试删除会话。如果清理失败，运维人员必须先按照 `README.md` 中的恢复流程操作，然后才能开始下一次运行。

在默认的 notebook 配置下，该单元格返回 `SKIPPED`，不执行任何实时操作。


In [ ]:
import json

agentcore_e2e_result = await run_managed_e2e()
if agentcore_e2e_result["result"] not in {"PASSED", "SKIPPED"}:
    print(json.dumps(agentcore_e2e_result, indent=2, sort_keys=True))
    raise RuntimeError(
        "Managed AgentCore test did not complete safely; review the "
        "sanitized report before any rerun."
    )

agentcore_e2e_result

### 解读支付与结算证据

托管运行的结果会记录：模型是否完成了工具调用、AgentCore Payments 是否生成了支付证明、商家是否返回了 HTTP `200`，以及临时会话是否已被删除。应用会对原始 HTTP 响应字节计算哈希，并且不会在报告中包含提供方凭据、支付请求头和内部标识符。

商家接受只能说明付费请求已完成。独立结算和交易最终性需要另外的证据。因此，本示例报告 `settlement_verified=false`。


In [ ]:
if agentcore_e2e_result["result"] == "PASSED":
    assert agentcore_e2e_result["model_run_completed"] is True
    assert agentcore_e2e_result["agentcore_payment_path_completed"] is True
    assert agentcore_e2e_result["merchant_paid_retry_completed"] is True
    assert agentcore_e2e_result["status_code"] == 200
    assert agentcore_e2e_result["payment_attempts"] == 1
    assert agentcore_e2e_result["session_cleanup"] == "DELETED"
    assert agentcore_e2e_result["identifiers_logged"] is False
    assert agentcore_e2e_result["settlement_verified"] is False

{
    "result": agentcore_e2e_result["result"],
    "session_cleanup": agentcore_e2e_result.get("session_cleanup", "NOT_CREATED"),
    "settlement_verified": agentcore_e2e_result.get("settlement_verified", False),
    "interpretation": (
        "Combined testnet path completed; settlement still needs separate evidence."
        if agentcore_e2e_result["result"] == "PASSED"
        else "Live path was not enabled; no testnet transaction was attempted."
    ),
}

### 可选的 CloudWatch 生命周期证据

这个公开的 notebook 不包含 CloudWatch 截图。对于经过审核的实时运行，请在运行前配置 AgentCore Payments 的日志投递，并且只将 CloudWatch 用作经过脱敏处理的生命周期证据。不要分享账户 ID、请求 ID、钱包数据、支付标识符或证明请求头。

生命周期事件有助于确认会话创建、支付处理和会话删除。但它们无法独立证明支付金额、区块链结算、交易最终性或生产就绪状态。


## 将支付工作流应用到其他服务

同样的结构可以支持其他付费数据源，包括高级搜索、制裁筛查、市场数据或供应商验证。每个集成都应在智能体能够请求购买之前，定义好已批准的商家、允许的用途、最高金额、审批要求和会话时长。

由模型来选择付费工具何时可能有助于完成用户任务。由应用代码来决定该请求是否获得授权，记录由此产生的交易证据，并将模型的最终响应与实际的工具结果进行核对。当工作流连接到已批准的 AWS 测试网会话时，AgentCore Payments 会再提供一道支出边界。


## 参考资料

- [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/)
- [通过 Amazon Bedrock 使用 OpenAI 模型](https://developers.openai.com/api/docs/guides/amazon-bedrock)
- [AWS 公告：能够进行交易的智能体](https://aws.amazon.com/blogs/machine-learning/agents-that-transact-introducing-amazon-bedrock-agentcore-payments-built-with-coinbase-and-stripe/)
- [AgentCore Payments 的工作原理](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-how-it-works.html)
- [AgentCore Payments 前提条件](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-prerequisites.html)
- [创建有边界的支付会话](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-create-session.html)
- [分离支付相关的 IAM 角色](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/payments-iam-roles.html)